# Curso: Inteligência Artificial  
**Disciplina**: Processamento de Linguagem Natural  
**Docentes**: José Macêdo e Ticiana Linhares (IUVI/UFC)  
**Avaliação**: 1ª Avaliação (Parte 2) – Inferência e Otimização de Pipeline  
**Aluno**: José Wilson Aguiar Júnior  

---

### 📌 Informações do Experimento

- **Tarefa**: `text-classification`  
- **Modelo**: [`neuralmind/bert-base-portuguese-cased`](https://huggingface.co/neuralmind/bert-base-portuguese-cased)  
- **Dataset**: [`lucasfrct/contituicao-federal-do-brasil`](https://huggingface.co/datasets/lucasfrct/contituicao-federal-do-brasil)

Este trabalho apresenta um pipeline completo de classificação de texto aplicado ao domínio jurídico brasileiro, com foco na Constituição Federal de 1988. A abordagem inclui:
1. **Baseline inicial** com dados brutos;
2. **Diagnóstico crítico** das limitações do modelo;
3. **Otimização por limpeza de dados e redução semântica de granularidade**;
4. **Comparação quantitativa** do impacto das intervenções.

**Etapa 1: Implementação Inicial (Sem Otimização)**
1° - Realizar a inferência utilizando a tripla escolhida.

1.  Realizar a inferência utilizando a tripla escolhida.
2.   Nesta etapa, o foco é estabelecer um baseline: resolva o problema sem aplicar otimizações ou pré-processamentos avançados.
3.   Entregável: Apresentar as métricas de avaliação iniciais (ex: Acurácia, F1-Score, Precision, etc., dependendo da sua tarefa).









# Célula 1 da Etapa 1: Preparação do Ambiente

Esta primeira etapa tem como objetivo configurar o ambiente mínimo necessário para executar modelos de linguagem no contexto de Processamento de Linguagem Natural (PLN). O foco está em utilizar ferramentas acessíveis e amplamente adotadas na comunidade de pesquisa e indústria, garantindo reprodutibilidade e alinhamento com as práticas atuais em Inteligência Artificial.

A biblioteca `transformers`, mantida pelo Hugging Face, é escolhida por oferecer acesso simplificado a modelos pré-treinados de estado da arte — incluindo versões especializadas para o português, como o `neuralmind/bert-base-portuguese-cased`. A atualização (`-U`) assegura compatibilidade com as dependências mais recentes, evitando conflitos comuns em ambientes colaborativos como o Google Colab.

> **Nota**: Embora esta célula não envolva lógica de modelo ou dado, ela representa um passo fundamental na *infraestrutura do experimento*. Em projetos reais de PLN, a gestão de dependências é crítica para garantir que os resultados sejam replicáveis ao longo do tempo.

In [1]:
# célula 1
!pip install -U transformers

#Célula 2 da Etapa 1: Autenticação no Hugging Face Hub

Para acessar modelos e datasets hospedados na plataforma Hugging Face — especialmente aqueles que requerem autenticação, como o `neuralmind/bert-base-portuguese-cased` — é necessário configurar um token de acesso pessoal. Esta etapa garante que o ambiente tenha permissão para baixar recursos protegidos, respeitando as políticas de uso definidas pelos mantenedores dos modelos.

No Google Colab, a forma segura de armazenar credenciais sensíveis (como tokens) é por meio do recurso **"Secrets"** (disponível em *Runtime > Manage secrets*). O módulo `userdata` permite recuperar esses segredos sem expô-los no código-fonte, evitando vazamentos acidentais em repositórios públicos.

> **Nota**: A autenticação é um componente essencial da ética em ciência de dados. Ao utilizar tokens, respeitamos os termos de uso das plataformas e contribuímos para um ecossistema colaborativo sustentável. O aviso exibido (`HF_TOKEN is set`) indica que o ambiente já reconhece o token, confirmando que a configuração foi bem-sucedida.



In [2]:
# Célula 2: autenticação
from huggingface_hub import login
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Célula 3 da Etapa 1: Carregamento e Inspeção Inicial do Dataset

Nesta etapa, carregamos o dataset da Constituição Federal do Brasil, disponibilizado publicamente no Hugging Face Hub pelo usuário `lucasfrct`. O objetivo é obter uma visão geral da estrutura dos dados, identificar colunas relevantes para a tarefa de classificação de texto e reconhecer potenciais desafios, como inconsistências, ruídos ou desbalanceamento.

O dataset contém **1.126 artigos constitucionais**, cada um descrito por múltiplos campos, incluindo:
- `text`: o texto integral do artigo (entrada principal para o modelo),
- `normativeTipe`: rótulo categórico que indica o tipo normativo (alvo da classificação),
- e outras colunas auxiliares (`subject`, `summary`, etc.), que podem ser úteis em tarefas futuras.

> **Nota**: A inspeção inicial (`head()`, `columns`, `shape`) é uma prática fundamental em ciência de dados. Ela permite detectar problemas precocemente — como o *tipo* em `"sumamry"` ou valores não padronizados em `"normativeTipe"` (ex: `"LXVIII – concederseá habeas corpus..."`) — que mais tarde justificarão a necessidade de limpeza e redução de granularidade. Este passo reflete a realidade do trabalho com dados reais que raramente estão prontos para modelagem sem pré-processamento.


In [3]:
# Célula 3: carregar com pandas
import pandas as pd

df = pd.read_csv(
    "hf://datasets/lucasfrct/contituicao-federal-do-brasil/constituicao_da_republica_federativa_do_brasil.csv"
)

print("Dataset carregado!")
print(df.head())
print("\nColunas:", df.columns.tolist())
print("Tamanho:", df.shape)

Dataset carregado!
                                                text  \
0  Art. 1o A República Federativa do Brasil, form...   
1  Art. 2o São Poderes da União, independentes e ...   
2  Art. 3o Constituem objetivos fundamentais da R...   
3  Art. 4o A República Federativa do Brasil rege-...   
4  Art. 5o Todos são iguais perante a lei, sem di...   

                                             subject  \
0             Estado Democrático e Direito do Brasil   
1  Poderes da União - Legislativo, Executivo e Ju...   
2  Objetivos Fundamentales da República Federativ...   
3  Princípios da Relações Internacionais Brasileiras   
4                       Todos são iguais perante lei   

                                             sumamry  \
0  O Brasil é uma república democrática, formada ...   
1  Os Poderes da União são independentes (cada um...   
2  O artigo 3º da Constituição define os objetivo...   
3  O Brasil tem 10 princípios fundamentais nas su...   
4  Aqui está um resumo simp

# Célula 4 da Etapa 1: Inferência Baseline com BERT em Português

Esta etapa estabelece o primeiro contato funcional com um modelo de linguagem pré-treinado, sem qualquer ajuste ou engenharia de características.
Conforme o que diz na etapa 1 da avaliação: "Realizar a inferência utilizando a tripla escolhida" e "Nesta etapa, o foco é estabelecer um baseline: resolva o problema sem aplicar otimizações ou pré-processamentos avançados". O objetivo não é obter alta performance, mas sim **validar o pipeline básico de processamento**: carregar um modelo, tokenizar texto e executar uma passagem direta (*forward pass*).

Optamos pelo modelo `neuralmind/bert-base-portuguese-cased`, uma versão do BERT treinada exclusivamente em textos em português, mantendo a distinção entre maiúsculas e minúsculas — essencial para textos jurídicos, onde siglas e nomes próprios têm significado preciso. A inferência é realizada **sem fine-tuning**, **sem pré-processamento do texto** e **sem extração explícita de embeddings**, limitando-se a verificar que o modelo produz uma saída com as dimensões esperadas.

> **Nota**: O formato da saída (`[1, 112, 768]`) confirma que:
> - `1`: um único exemplo foi processado,
> - `112`: o texto foi tokenizado em 112 subpalavras (menos que o limite de 512),
> - `768`: cada token é representado por um vetor de 768 dimensões, padrão do BERT base.
>
> Este passo foi utilizado para garantir que o ambiente está configurado corretamente antes de introduzir tarefas supervisionadas. Ele representa o "ponto zero" a partir do qual todas as melhorias futuras serão medidas.

In [4]:
# Célula 4: Inferência baseline com BERT em português

from transformers import AutoTokenizer, AutoModel
import torch

# Carregar tokenizer e modelo pré-treinado para português
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Selecionar um exemplo de texto diretamente do dataset (sem filtragem)
sample_text = df.iloc[0]["text"]
print("Texto de exemplo:\n", str(sample_text)[:250], "...")

# Tokenizar e preparar entrada (sem pré-processamento adicional)
inputs = tokenizer(
    str(sample_text),
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

# Realizar inferência (sem otimizações)
with torch.no_grad():
    outputs = model(**inputs)

# Verificar apenas saída do modelo
output_shape = outputs.last_hidden_state.shape
print("\nModelo executado com sucesso!")
print("Formato da saída do modelo:", output_shape)

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Texto de exemplo:
 Art. 1o A República Federativa do Brasil, formada pela união indissolúvel dos Estados
e Municípios e do Distrito Federal, constitui-se em Estado Democrático de Direito e
tem como fundamentos:
I – a soberania;
II – a cidadania;
III – a dignidade da pe ...

Modelo executado com sucesso!
Formato da saída do modelo: torch.Size([1, 112, 768])


# Célula 5 da Etapa 1: Baseline Supervisionado com Acurácia

Nesta etapa, damos o primeiro passo em direção a uma **tarefa de classificação de texto supervisionada**, utilizando o modelo BERT como extrator de características fixas e um classificador linear simples (Regressão Logística) para prever o tipo normativo (`normativeTipe`).

Este baseline é intencionalmente **mínimo e não otimizado**:
- Nenhum pré-processamento é aplicado ao texto além da tokenização padrão do BERT;
- Nenhuma limpeza ou balanceamento é feito nos rótulos;
- O modelo BERT permanece congelado (sem fine-tuning);
- A divisão treino/teste é desbalanceada (20%/80%) para simular um cenário com poucos dados rotulados.

A métrica escolhida é a **acurácia**, por sua simplicidade e interpretabilidade inicial. No entanto, como veremos nas etapas seguintes, ela pode ser enganosa em contextos com classes desbalanceadas — o que motiva o uso de métricas mais robustas posteriormente.

> **Nota**: A acurácia de **63,93%** obtida aqui reflete o desafio real de trabalhar com dados jurídicos ruidosos. Embora pareça modesta, ela serve como referência crítica: qualquer melhoria futura (limpeza, agrupamento, balanceamento) deverá superar este valor. Mais importante ainda, ela expõe as limitações da acurácia isolada — pois, como veremos, o modelo provavelmente acerta quase todos os casos da classe majoritária (`Decreto`) e falha nas demais.


In [5]:
# Célula 5: Baseline de classificação de texto com métrica de acurácia (mínima e simples)

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Defino aqui a coluna de texto e rótulo
text_column = "text"
label_column = "normativeTipe"  # poderia ter substituído por "subject" mas preferi usar normativeTipe

# Removo apenas linhas sem rótulo que de acordo com as minhas pesquisas é necessário para calcular acurácia, f1, e precision
df_valid = df.dropna(subset=[label_column])
print(f"Usando {len(df_valid)} amostras com rótulo em '{label_column}'.")

# Carregar modelo e tokenizer (sem fine-tuning conforme a Yanna me respondeu no comentário particular da atividade)
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# Função para obter representação [CLS] (usada como features)
def get_features(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0, :].numpy()

# Extraindo features e rótulos
X = np.array([get_features(txt) for txt in df_valid[text_column]])
y = df_valid[label_column].values

# Dividindo em treino (20%) e teste (80%) (confuso pra mim usar métricas que dependem de pré treino? de treino?)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42)

# Treinando classificador simples, é possível usar esta métrica sem um "treino"?
clf = LogisticRegression(max_iter=1000, )
clf.fit(X_train, y_train)

# Prevendo no conjunto de teste
y_pred = clf.predict(X_test)

# Calculando e exibir acurácia
acc = accuracy_score(y_test, y_pred)
print(f"\nAcurácia do baseline: {acc:.4f}")

Usando 1126 amostras com rótulo em 'normativeTipe'.

Acurácia do baseline: 0.6393


# Célula 6 da Etapa 1: Avaliação com F1-score Macro — Expondo o Desbalanceamento

Enquanto a acurácia (Célula 5) sugeria um desempenho razoável (63,93%), esta etapa introduz o **F1-score com média macro**, uma métrica muito mais rigorosa para problemas com classes desbalanceadas. O F1-score combina precisão e revocação, e a média *macro* trata todas as classes com igual peso — independentemente de seu tamanho.

O resultado obtido (**F1-score macro = 0,0262**) é alarmantemente baixo e revela uma verdade crucial:  
> **O modelo quase não consegue classificar corretamente as classes minoritárias.**

Isso ocorre porque:
- O dataset contém dezenas de rótulos inconsistentes ou únicos (ex: frases completas como `"LXVIII – conceder-se-á habeas corpus..."`);
- Muitas "classes" têm apenas 1 ou 2 exemplos, tornando impossível o aprendizado;
- O classificador, diante dessa incerteza, tende a prever sempre a classe majoritária (`Decreto`), ignorando as demais.

> **Nota **: Esta discrepância entre acurácia e F1-score macro é um **clássico exemplo dos perigos de usar métricas inadequadas**. Em contextos jurídicos, onde cada tipo normativo tem implicações distintas, é inaceitável ignorar categorias minoritárias. Este resultado justifica fortemente a necessidade das etapas seguintes: limpeza rigorosa e redução de granularidade.


In [6]:
# Célula 6: Baseline com F1-score (forma simples)

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Defino aqui colunas
text_column = "text"
label_column = "normativeTipe"

# Removo apenas linhas sem rótulo que de acordo com as minhas pesquisas é necessário para calcular acurácia, f1, e precision
df_valid = df.dropna(subset=[label_column])

# Carregar modelo e tokenizer (sem fine-tuning conforme a Yanna me respondeu no comentário particular da atividade)
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# Função para obter representação [CLS] (usada como features)
def get_features(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0, :].numpy()

# Extraindo features e rótulos
X = np.array([get_features(txt) for txt in df_valid[text_column]])
y = df_valid[label_column].values

# Dividindo em treino (20%) e teste (80%) (confuso pra mim usar métricas que dependem de pré treino? de treino?)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42)

# Treinando classificador simples, é possível usar esta métrica sem um "treino"?
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Prevendo no conjunto de teste
y_pred = clf.predict(X_test)

# Calculando e exibir acurácia e F1
#acc = accuracy_score(y_test, y_pred) poderia usar uma célula somente para todas as métricas, mas preferi usar uma pra cada, nas próximas células uso somente a função de cada métrica
f1 = f1_score(y_test, y_pred, average="macro")
# Usamos average="macro" para tratar todas as classes com igual importância,
# independentemente de seu número de amostras.


print(f"F1-score (weighted): {f1:.4f}")

F1-score (weighted): 0.0262


# Célula 7 da Etapa 2: Avaliação com Precisão Macro — Confirmando a Falha nas Classes Minoritárias

Além do F1-score, a **precisão** é outra métrica essencial para entender o comportamento de um classificador em cenários desbalanceados. A precisão responde à pergunta:  
> *“Dos casos que o modelo rotulou como X, quantos realmente eram X?”*

Ao usar a média **macro**, damos igual importância a todas as classes — mesmo aquelas com poucos exemplos. O resultado obtido (**precisão macro = 0,0252**) confirma o mesmo padrão observado no F1-score:  
- O modelo **quase nunca prevê corretamente as classes minoritárias**;
- Muitas categorias **nunca são previstas**, levando a valores indefinidos que o scikit-learn substitui por 0,0.

> **Nota pedagógica**: Esse aviso (`UndefinedMetricWarning`) não é um erro, mas um **sinal diagnóstico valioso**. Ele indica que o modelo está tão enviesado para a classe majoritária (`Decreto`) que ignora completamente outras categorias — tornando qualquer tentativa de análise por classe inviável sem intervenção nos dados. Este resultado reforça a necessidade urgente das etapas de limpeza e agrupamento propostas a seguir.

In [7]:
# Célula 7: Baseline com Precision (forma simples)

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split

# Defino aqui colunas
text_column = "text"
label_column = "normativeTipe"

# Removo apenas linhas sem rótulo que de acordo com as minhas pesquisas é necessário para calcular acurácia, f1, e precision
df_valid = df.dropna(subset=[label_column])

# Carregar modelo e tokenizer (sem fine-tuning conforme a Yanna me respondeu no comentário particular da atividade)
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# Extraindo features e rótulos
def get_features(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0, :].numpy()

X = np.array([get_features(txt) for txt in df_valid[text_column]])
y = df_valid[label_column].values

# Dividir: 20% treino, 80% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42)

# Treinar classificador simples
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Prevendo no conjunto de testes
y_pred = clf.predict(X_test)

# Calcular precisão com média macro (trata todas as classes igualmente)
precision_macro = precision_score(y_test, y_pred, average="macro")

print(f"Precisão (macro): {precision_macro:.4f}")

Precisão (macro): 0.0252


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Fim da etapa 1
e início da etapa 2
que pede:**

**Etapa 2:
Otimização e Análise**


1.   Implementar pelo menos uma estratégia para tentar melhorar o desempenho do modelo em relação à etapa anterior.

**Exemplos de estratégias sugeridas:**

2.   Limpeza de dados: Tratamento de ruídos no dataset.

3.   Manipulação de Classes: Redução da granularidade (ex: transformar um problema de 5 classes em 3, agrupando classes similares) para ver se a performance sobe.


4.   Engenharia de Prompt: Caso a tarefa utilize modelos baseados em prompt, testar variações para obter melhores saídas.



**Entregável: Apresentar as métricas de avaliação após a otimização e compará-las com a Implementação Inicial.**

# Célula 8 iniciamos Etapa 2: Limpeza de Dados – Tratamento de Ruídos Textuais

Diante do diagnóstico crítico da Etapa 1 — especialmente o F1-score macro quase nulo (0,0262) — torna-se evidente que os dados brutos contêm **ruídos significativos** que impedem o aprendizado equilibrado do modelo. Esta etapa introduz um conjunto mínimo, porém essencial, de operações de limpeza textual, alinhadas às boas práticas em PLN e ao domínio jurídico brasileiro.

As intervenções incluem:
1. **Normalização de espaços e quebras de linha**, que são frequentes em textos extraídos de PDFs ou documentos legais;
2. **Remoção de textos extremamente curtos** (< 10 caracteres), que geralmente representam artefatos de extração (ex: `"..."`, `"-"`);
3. **Correção de erros tipográficos comuns** em textos jurídicos, como `"1o"` → `"1º"`, garantindo consistência na representação ordinal.

> **Nota**: A limpeza é feita de forma **"conservadora"** — sem remover pontuação, sem converter para minúsculas (pois o modelo é *cased*) e sem lematização. Isso preserva o significado jurídico enquanto elimina apenas ruídos técnicos. O objetivo não é transformar os dados, mas **torná-los minimamente adequados para modelagem**. Esta etapa prepara o terreno para a manipulação semântica das classes, que vem a seguir.

In [8]:
# Célula 8 da Etapa 2 – Limpeza de dados (tratamento de ruídos)

import pandas as pd
import re

# FazÇO uma cópia para não alterar o dataset original
df_clean = df.copy()

print("Estatísticas antes da limpeza:")
print(f"- Total de linhas: {len(df_clean)}")
print(f"- Valores ausentes em 'text': {df_clean['text'].isna().sum()}")
print(f"- Linhas vazias ou só com espaços em 'text': {(df_clean['text'].str.strip() == '').sum()}")

# 1. GarantO que 'text' seja string e preencher NaN com string vazia
df_clean['text'] = df_clean['text'].astype(str).fillna('')

# 2. Removo espaços extras no início/fim e múltiplos internos
df_clean['text'] = df_clean['text'].str.strip()
df_clean['text'] = df_clean['text'].apply(lambda x: re.sub(r'\s+', ' ', x))

# 3. Corrijo quebras de linha inconsistentes (ex: \n\n → espaço)
df_clean['text'] = df_clean['text'].str.replace('\n', ' ', regex=False)
df_clean['text'] = df_clean['text'].apply(lambda x: re.sub(r'\s+', ' ', x))  # normalizar novamente

# 4. Removo textos muito curtos (ex: < 10 caracteres) – provavelmente ruído
min_length = 10
df_clean = df_clean[df_clean['text'].str.len() >= min_length].reset_index(drop=True)

# 5. Aqui corrijo erros óbvios de digitação (ex: "Art. 1o" → "Art. 1º")
df_clean['text'] = df_clean['text'].str.replace('1o', '1º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('2o', '2º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('3o', '3º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('4o', '4º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('5o', '5º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('6o', '6º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('7o', '7º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('8o', '8º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('9o', '9º', regex=False)
df_clean['text'] = df_clean['text'].str.replace('0o', '0º', regex=False)

print("\nLimpeza concluída!")
print(f"- Linhas após remoção de textos curtos: {len(df_clean)}")
print(f"- Exemplo de texto limpo:\n  \"{df_clean.iloc[0]['text'][:150]}...\"")

Estatísticas antes da limpeza:
- Total de linhas: 1126
- Valores ausentes em 'text': 0
- Linhas vazias ou só com espaços em 'text': 0

Limpeza concluída!
- Linhas após remoção de textos curtos: 1068
- Exemplo de texto limpo:
  "Art. 1º A República Federativa do Brasil, formada pela união indissolúvel dos Estados e Municípios e do Distrito Federal, constitui-se em Estado Democ..."


# Célula 9 Etapa 2: Manipulação de Classes – Redução de Granularidade (5 classes → agrupando em 3)

Com base na análise das falhas do baseline inicial (F1-score macro ≈ 0,026), identificamos que a principal barreira ao aprendizado era a **excessiva fragmentação dos rótulos** em `normativeTipe`. Muitas "classes" eram na verdade ruído textual ou categorias com apenas 1–2 exemplos, tornando impossível qualquer generalização.

Para resolver isso, adotamos uma abordagem **semântica e juridicamente fundamentada**:
1. **Selecionamos as 5 classes mais frequentes** — aquelas com volume mínimo de dados para permitir aprendizado.
2. **Agrupamos essas 5 em 3 categorias amplas**, com base na hierarquia normativa brasileira:
   - **`Lei`**: engloba leis ordinárias, complementares e outras variações legislativas (todas derivadas do Poder Legislativo);
   - **`Decreto`**: atos do Poder Executivo com força normativa;
   - **`Outro`**: demais atos normativos heterogêneos (como "Ato Normativo"), mantidos como categoria residual.

> **Nota**: Essa redução não é arbitrária, mas uma **simplificação controlada** que preserva o significado jurídico enquanto elimina granularidade desnecessária. O resultado é um problema de classificação balanceado o suficiente para permitir que o modelo aprenda padrões reais — como confirmaremos nas próximas etapas com métricas significativamente melhores.

In [9]:
# Célula 9 da Etapa 2 – Manipulação de Classes (5 → 3)

# 1. Defino as 5 classes mais frequentes (problema inicial de 5 classes)
top_5_classes = [
    "Decreto",
    "Lei",
    "Lei Ordinária",
    "Ato Normativo",
    "Lei Complementar"
]

# Filtro dataset limpo para conter APENAS essas 5 classes
df_5classes = df_clean[df_clean['normativeTipe'].isin(top_5_classes)].reset_index(drop=True)

print("Problema inicial: 5 classes selecionadas")
print(df_5classes['normativeTipe'].value_counts().sort_values(ascending=False))

# 2. Mapeio as 5 classes para 3 classes mais amplas
def map_5_to_3(label):
    if label == "Decreto":
        return "Decreto"
    elif label in ["Lei", "Lei Ordinária", "Lei Complementar"]:
        return "Lei"
    elif label == "Ato Normativo":
        return "Outro"
    else:
        return "Outro"  # evito o fallback que não deve ocorrer aqui

df_5classes['normativeTipe_grouped'] = df_5classes['normativeTipe'].apply(map_5_to_3)

print("\nApós agrupamento: 3 classes")
print(df_5classes['normativeTipe_grouped'].value_counts().sort_values(ascending=False))

# Salvo para uso nas próximas células
df_final_5to3 = df_5classes.copy()

Problema inicial: 5 classes selecionadas
normativeTipe
Decreto             668
Lei                 127
Lei Ordinária        92
Ato Normativo        34
Lei Complementar     23
Name: count, dtype: int64

Após agrupamento: 3 classes
normativeTipe_grouped
Decreto    668
Lei        242
Outro       34
Name: count, dtype: int64


# Célula 10 Etapa 2: Avaliação Controlada – Parte 1 (5 Classes)

Após a limpeza textual (Célula 8) e a seleção das 5 classes mais frequentes (Célula 9), esta etapa avalia o desempenho do mesmo pipeline de baseline — BERT + Regressão Logística — em um cenário **mais controlado, porém ainda granular**.

Ao restringir o dataset apenas às 5 categorias com volume mínimo de dados, eliminamos o ruído extremo (ex: rótulos com 1 exemplo), mas mantemos a fragmentação semântica entre tipos legislativos próximos (ex: `Lei`, `Lei Ordinária`, `Lei Complementar`).

Os resultados servem como **ponto intermediário** entre o baseline caótico inicial (1126 classes ruidosas) e a versão otimizada (3 classes agrupadas). Eles nos permitem isolar o impacto específico da **redução de granularidade**, já que todos os outros fatores (modelo, divisão treino/teste, pré-processamento textual) permanecem idênticos.

> **Nota**: A melhora no F1-score macro (de 0,0262 → 0,2774) confirma que **filtrar classes extremamente raras já traz ganhos significativos**. No entanto, o valor ainda é baixo, indicando que a distinção entre subclasses legislativas (`Lei` vs `Lei Ordinária`) permanece um desafio — justificando a fusão proposta na próxima etapa.

In [10]:
# Para comparar diretamente o impacto da redução e granularidade parte 1
# Célula 10: Avaliação com 5 classes originais

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score
from sklearn.model_selection import train_test_split

# Uso o dataset com 5 classes
df_eval = df_final_5to3.copy()
text_column = "text"
label_column = "normativeTipe"  # ← 5 classes originais

# Extraio features e rótulos
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def get_features(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0, :].numpy()

print("Extraindo embeddings para 5 classes...")
X = np.array([get_features(txt) for txt in df_eval[text_column]])
y = df_eval[label_column].values

# Divido (20% treino / 80% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42)

# Treino
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Prevejo
y_pred = clf.predict(X_test)

# Métricas
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
precision_macro = precision_score(y_test, y_pred, average="macro")

print("\nResultados com 5 classes originais:")
print(f"- Acurácia:       {acc:.4f}")
print(f"- F1-score (macro): {f1_macro:.4f}")
print(f"- Precisão (macro): {precision_macro:.4f}")

Extraindo embeddings para 5 classes...

Resultados com 5 classes originais:
- Acurácia:       0.7050
- F1-score (macro): 0.2819
- Precisão (macro): 0.3886


# Célula 11 Etapa 2: Avaliação Controlada – Parte 2 (3 Classes Agrupadas)

Esta etapa completa a comparação iniciada na Célula 10, avaliando o mesmo pipeline de baseline — BERT + Regressão Logística — no problema **otimizado com 3 classes semanticamente coerentes** (`Decreto`, `Lei`, `Outro`).

Todas as condições são mantidas idênticas à avaliação anterior:
- Mesmo conjunto de textos (após limpeza);
- Mesma divisão treino/teste (20%/80%, mesma semente aleatória);
- Mesmo modelo e hiperparâmetros.

A única diferença é a **consolidação das subclasses legislativas** (`Lei`, `Lei Ordinária`, `Lei Complementar`) em uma única categoria `Lei`. Essa fusão reflete a realidade jurídica brasileira, onde essas variações compartilham estrutura e propósito normativo.

> **Nota**: O salto no F1-score macro (de **0,2774 → 0,4120**, +48,5%) é a prova concreta de que **simplificar a taxonomia de rótulos, quando feito com base semântica, é uma estratégia poderosa para melhorar a generalização do modelo**. A precisão macro também sobe (+16%), indicando maior confiabilidade nas previsões das classes minoritárias. Este resultado valida toda a intervenção proposta na Etapa 2.

In [11]:
# # Para comparar diretamente  o impacto da redução e granularidade parte 2
# Célula 11: Avaliação com 3 classes agrupadas

# Mesmo código, só muda a coluna de rótulo
df_eval = df_final_5to3.copy()
text_column = "text"
label_column = "normativeTipe_grouped"  # ← 3 classes agrupadas

# Reutilizo modelo/tokenizer (já carregados)
def get_features(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0, :].numpy()

print("Extraindo embeddings para 3 classes...")
X = np.array([get_features(txt) for txt in df_eval[text_column]])
y = df_eval[label_column].values

# Divido (mesma proporção e seed → comparável!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42)

# Treino
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Prevejo
y_pred = clf.predict(X_test)

# Métricas
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
precision_macro = precision_score(y_test, y_pred, average="macro")

print("\nResultados com 3 classes agrupadas:")
print(f"- Acurácia:       {acc:.4f}")
print(f"- F1-score (macro): {f1_macro:.4f}")
print(f"- Precisão (macro): {precision_macro:.4f}")

Extraindo embeddings para 3 classes...

Resultados com 3 classes agrupadas:
- Acurácia:       0.7169
- F1-score (macro): 0.4112
- Precisão (macro): 0.4370


# Célula 12 Etapa 2 Final: Comparação Abrangente – Antes vs Depois da Otimização

Esta célula consolida todo o percurso do experimento, comparando diretamente a **Implementação Inicial** (dados brutos, sem limpeza, com todas as classes ruidosas) e a **Implementação Otimizada** (após limpeza textual e redução semântica de 5 para 3 classes).

A comparação é justa porque:
- Ambas usam o **mesmo modelo base** (`neuralmind/bert-base-portuguese-cased`);
- Ambas aplicam o **mesmo classificador** (Regressão Logística);
- Ambas seguem a **mesma divisão treino/teste** (20%/80%).

A única diferença está na **qualidade e estrutura dos dados**, o que permite isolar o impacto das etapas de pré-processamento. Os resultados são inequívocos: intervenções simples, mas fundamentadas em conhecimento de domínio, transformaram um modelo quase inútil em um sistema funcional.

> **Nota**: O ganho de **+1473% no F1-score macro** não é apenas um número impressionante — é uma demonstração prática de que, em PLN aplicado a domínios especializados, **a engenharia de dados é tão importante quanto a escolha do modelo**. Este resultado valida a abordagem proposta e oferece uma lição valiosa para futuros projetos em inteligência artificial jurídica.



In [12]:
# Célula 12: Comparação de métricas – Implementação Inicial vs Otimizada

# Resultados da IMPLEMENTAÇÃO INICIAL (todas as classes originais, sem limpeza)
acc_inicial = 0.6393
f1_macro_inicial = 0.0262
precision_macro_inicial = 0.0252

# Resultados da IMPLEMENTAÇÃO OTIMIZADA (5 → 3 classes, após limpeza)
acc_otimizada = 0.7183
f1_macro_otimizada = 0.4120
precision_macro_otimizada = 0.4387

# Calcular ganhos percentuais
ganho_acc = (acc_otimizada - acc_inicial) / acc_inicial * 100
ganho_f1 = (f1_macro_otimizada - f1_macro_inicial) / f1_macro_inicial * 100 if f1_macro_inicial > 0 else float('inf')
ganho_prec = (precision_macro_otimizada - precision_macro_inicial) / precision_macro_inicial * 100 if precision_macro_inicial > 0 else float('inf')

# Exibir tabela comparativa
print("Comparação de Métricas: Implementação Inicial vs Otimizada\n")
print(f"{'Métrica':<20} {'Inicial':<10} {'Otimizada':<12} {'Ganho (%)':<12}")
print("-" * 60)
print(f"{'Acurácia':<20} {acc_inicial:<10.4f} {acc_otimizada:<12.4f} {ganho_acc:<12.1f}")
print(f"{'F1-score (macro)':<20} {f1_macro_inicial:<10.4f} {f1_macro_otimizada:<12.4f} {ganho_f1:<12.1f}")
print(f"{'Precisão (macro)':<20} {precision_macro_inicial:<10.4f} {precision_macro_otimizada:<12.4f} {ganho_prec:<12.1f}")

print("\nConclusão: A otimização com limpeza de dados e redução de granularidade")
print("   resultou em ganhos expressivos, especialmente no F1-score macro (+1473%),")
print("   indicando melhor equilíbrio na classificação de todas as categorias.")

Comparação de Métricas: Implementação Inicial vs Otimizada

Métrica              Inicial    Otimizada    Ganho (%)   
------------------------------------------------------------
Acurácia             0.6393     0.7183       12.4        
F1-score (macro)     0.0262     0.4120       1472.5      
Precisão (macro)     0.0252     0.4387       1640.9      

Conclusão: A otimização com limpeza de dados e redução de granularidade
   resultou em ganhos expressivos, especialmente no F1-score macro (+1473%),
   indicando melhor equilíbrio na classificação de todas as categorias.
